In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/students_dropout_academic_success.csv', sep=',')

# Basic info
print("Shape of dataset:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Shape of dataset: (4424, 37)

Column names:
['Marital Status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inf

In [3]:
# Check target column distribution
print(df['target'].value_counts())
print("\nPercentage:")
print(df['target'].value_counts(normalize=True) * 100)

target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Percentage:
target
Graduate    49.932188
Dropout     32.120253
Enrolled    17.947559
Name: proportion, dtype: float64


In [4]:
# Convert target to numeric for analysis (Dropout = 1, others = 0)
df['is_dropout'] = (df['target'] == 'Dropout').astype(int)

# Check dropout rate by key categorical factors
print("Dropout rate by Scholarship holder:")
print(df.groupby('Scholarship holder')['is_dropout'].mean() * 100)

print("\nDropout rate by Debtor status:")
print(df.groupby('Debtor')['is_dropout'].mean() * 100)

print("\nDropout rate by Tuition fees up to date:")
print(df.groupby('Tuition fees up to date')['is_dropout'].mean() * 100)

print("\nDropout rate by Gender:")
print(df.groupby('Gender')['is_dropout'].mean() * 100)

Dropout rate by Scholarship holder:
Scholarship holder
0    38.706767
1    12.192903
Name: is_dropout, dtype: float64

Dropout rate by Debtor status:
Debtor
0    28.283601
1    62.027833
Name: is_dropout, dtype: float64

Dropout rate by Tuition fees up to date:
Tuition fees up to date
0    86.553030
1    24.743326
Name: is_dropout, dtype: float64

Dropout rate by Gender:
Gender
0    25.104603
1    45.051414
Name: is_dropout, dtype: float64


In [5]:
# Dropout rate by academic performance (semester 1 approved units)
print("Average curricular units approved (1st sem) by dropout status:")
print(df.groupby('target')['Curricular units 1st sem (approved)'].mean())

print("\nAverage curricular units approved (2nd sem) by dropout status:")
print(df.groupby('target')['Curricular units 2nd sem (approved)'].mean())

print("\nAverage 1st sem grade by dropout status:")
print(df.groupby('target')['Curricular units 1st sem (grade)'].mean())

Average curricular units approved (1st sem) by dropout status:
target
Dropout     2.551724
Enrolled    4.318640
Graduate    6.232232
Name: Curricular units 1st sem (approved), dtype: float64

Average curricular units approved (2nd sem) by dropout status:
target
Dropout     1.940183
Enrolled    4.057935
Graduate    6.177003
Name: Curricular units 2nd sem (approved), dtype: float64

Average 1st sem grade by dropout status:
target
Dropout      7.256656
Enrolled    11.125257
Graduate    12.643655
Name: Curricular units 1st sem (grade), dtype: float64


In [6]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum().sum())  # Total missing values across dataset

# Check for duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Check data types
print("\nData types summary:")
print(df.dtypes.value_counts())

Missing values per column:
0

Duplicate rows: 0

Data types summary:
int64      30
float64     7
str         1
Name: count, dtype: int64


In [7]:
# Save cleaned dataset for SQL use
df.to_csv('../data/cleaned_student_data.csv', index=False)
print("Cleaned data saved successfully!")
print("Final shape:", df.shape)

Cleaned data saved successfully!
Final shape: (4424, 38)


In [8]:
import sqlite3

# Create SQLite database and load data
conn = sqlite3.connect('../sql/student_dropout.db')
df.to_sql('students', conn, if_exists='replace', index=False)

print("Database created successfully!")
print("Table 'students' loaded with", len(df), "rows")

conn.close()

Database created successfully!
Table 'students' loaded with 4424 rows


In [9]:
query1 = """
SELECT 
    Course,
    COUNT(*) as total_students,
    SUM(CASE WHEN target = 'Dropout' THEN 1 ELSE 0 END) as dropouts,
    ROUND(SUM(CASE WHEN target = 'Dropout' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as dropout_rate_pct
FROM students
GROUP BY Course
ORDER BY dropout_rate_pct DESC;
"""

conn = sqlite3.connect('../sql/student_dropout.db')
result1 = pd.read_sql_query(query1, conn)
conn.close()

result1

,Course,total_students,dropouts,dropout_rate_pct
0,33,12,8,66.67
1,9130,141,78,55.32
2,9119,170,92,54.12
3,9991,268,136,50.75
4,9853,192,85,44.27
5,9003,210,86,40.95
6,9556,86,33,38.37
7,171,215,82,38.14
8,9254,252,96,38.10
9,9670,268,95,35.45


In [10]:
query2 = """
SELECT 
    "Scholarship holder",
    "Debtor",
    "Tuition fees up to date",
    COUNT(*) as total_students,
    ROUND(SUM(CASE WHEN target = 'Dropout' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as dropout_rate_pct
FROM students
GROUP BY "Scholarship holder", "Debtor", "Tuition fees up to date"
ORDER BY dropout_rate_pct DESC;
"""

conn = sqlite3.connect('../sql/student_dropout.db')
result2 = pd.read_sql_query(query2, conn)
conn.close()

result2

,Scholarship holder,Debtor,Tuition fees up to date,total_students,dropout_rate_pct
0,0,0,0,259,89.19
1,0,1,0,223,88.79
2,1,1,0,23,73.91
3,1,0,0,23,47.83
4,0,1,1,196,42.35
5,0,0,1,2647,29.28
6,1,1,1,61,22.95
7,1,0,1,992,9.27


In [11]:
query3 = """
WITH risk_summary AS (
    SELECT 
        "Scholarship holder",
        "Tuition fees up to date",
        COUNT(*) as total_students,
        ROUND(SUM(CASE WHEN target = 'Dropout' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as dropout_rate_pct
    FROM students
    GROUP BY "Scholarship holder", "Tuition fees up to date"
)
SELECT * FROM risk_summary
WHERE dropout_rate_pct > 30
ORDER BY dropout_rate_pct DESC;
"""

conn = sqlite3.connect('../sql/student_dropout.db')
result3 = pd.read_sql_query(query3, conn)
conn.close()

result3

,Scholarship holder,Tuition fees up to date,total_students,dropout_rate_pct
0,0,0,482,89.00
1,1,0,46,60.87
2,0,1,2843,30.18


In [13]:
query4 = """
SELECT 
    Course,
    COUNT(*) as total_students,
    ROUND(SUM(CASE WHEN target = 'Dropout' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as dropout_rate_pct,
    RANK() OVER (ORDER BY ROUND(SUM(CASE WHEN target = 'Dropout' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) DESC) as risk_rank
FROM students
GROUP BY Course
HAVING COUNT(*) >= 100;
"""

conn = sqlite3.connect('../sql/student_dropout.db')
result4 = pd.read_sql_query(query4, conn)
conn.close()

result4

,Course,total_students,dropout_rate_pct,risk_rank
0,9130,141,55.32,1
1,9119,170,54.12,2
2,9991,268,50.75,3
3,9853,192,44.27,4
4,9003,210,40.95,5
5,171,215,38.14,6
6,9254,252,38.10,7
7,9670,268,35.45,8
8,9147,380,35.26,9
9,8014,215,33.02,10


In [14]:
# Step 1: Create a small reference table for risk categories
risk_categories = pd.DataFrame({
    'Scholarship holder': [0, 0, 1, 1],
    'Tuition fees up to date': [0, 1, 0, 1],
    'Risk Category': ['Critical', 'Moderate', 'High', 'Low']
})

conn = sqlite3.connect('../sql/student_dropout.db')
risk_categories.to_sql('risk_lookup', conn, if_exists='replace', index=False)
conn.close()

print("Risk lookup table created!")

Risk lookup table created!


In [15]:
query5 = """
SELECT 
    r."Risk Category",
    COUNT(*) as total_students,
    ROUND(SUM(CASE WHEN s.target = 'Dropout' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as dropout_rate_pct
FROM students s
JOIN risk_lookup r
    ON s."Scholarship holder" = r."Scholarship holder"
    AND s."Tuition fees up to date" = r."Tuition fees up to date"
GROUP BY r."Risk Category"
ORDER BY dropout_rate_pct DESC;
"""

conn = sqlite3.connect('../sql/student_dropout.db')
result5 = pd.read_sql_query(query5, conn)
conn.close()

result5

,Risk Category,total_students,dropout_rate_pct
0,Critical,482,89.00
1,High,46,60.87
2,Moderate,2843,30.18
3,Low,1053,10.07


In [16]:
query6 = """
SELECT 
    CASE 
        WHEN "Age at enrollment" <= 19 THEN 'Teen (≤19)'
        WHEN "Age at enrollment" BETWEEN 20 AND 25 THEN 'Young Adult (20-25)'
        ELSE 'Mature (26+)'
    END as age_group,
    COUNT(*) as total_students,
    ROUND(SUM(CASE WHEN target = 'Dropout' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as dropout_rate_pct
FROM students
GROUP BY age_group
ORDER BY dropout_rate_pct DESC;
"""

conn = sqlite3.connect('../sql/student_dropout.db')
result6 = pd.read_sql_query(query6, conn)
conn.close()

result6

,age_group,total_students,dropout_rate_pct
0,Mature (26+),1045,55.89
1,Young Adult (20-25),1427,29.99
2,Teen (≤19),1952,20.95


In [17]:
query7 = """
SELECT 
    target,
    ROUND(AVG("Admission grade"), 2) as avg_admission_grade,
    ROUND(AVG("Curricular units 1st sem (grade)"), 2) as avg_1st_sem_grade,
    ROUND(AVG("Unemployment rate"), 2) as avg_unemployment_rate
FROM students
GROUP BY target;
"""

conn = sqlite3.connect('../sql/student_dropout.db')
result7 = pd.read_sql_query(query7, conn)
conn.close()

result7

,target,avg_admission_grade,avg_1st_sem_grade,avg_unemployment_rate
0,Dropout,124.96,7.26,11.62
1,Enrolled,125.53,11.13,11.27
2,Graduate,128.79,12.64,11.64
